In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Chemin du CSV depuis le notebook
fichier = Path("../data/24_juillet_2025_donnees_preparees.csv")

# Charger les données
df = pd.read_csv(fichier)

# Repérer les 16 colonnes de pluie
colonnes_pluie = list(df.columns[1:17])

print("Nombre de lignes :", len(df))
print("Nombre de pluviomètres :", len(colonnes_pluie))
print(df.head())

Calculer les indicateurs

In [ ]:
# Les 16 mesures de pluie
pluie = df[colonnes_pluie]

# Intensité maximale parmi les stations
df["intensite_max"] = pluie.max(axis=1)

# Nombre de stations avec une pluie positive
df["stations_actives"] = pluie.gt(0).sum(axis=1)

# Nombre de stations avec une mesure disponible
df["stations_valides"] = pluie.notna().sum(axis=1)

# Proportion de stations actives
# Cette proportion constitue l'indicateur d'étendue spatiale de la pluie.
df["proportion_active"] = (
    df["stations_actives"] / df["stations_valides"]
)

print(df[[
    "intensite_max",
    "stations_actives",
    "proportion_active"
]].describe())

Le son STRAUSS
Le mapping :
Donnée Python	    Paramètre STRAUSS
intensite	        pitch_shift
etendue	            volume
Horodatage	        time_evo

#### Le mapping choisi
| Donnée météorologique | Paramètre STRAUSS | Effet recherché |
|------------------------|-------------------|-----------------|
| Intensité maximale | `pitch_shift` | Modifier la hauteur du son |
| Proportion de stations actives | `volume` | Modifier le volume sonore |
| Horodatage | `time_evo` | Organiser l'évolution dans le temps |

Lorsque la pluie devient plus intense, la hauteur du son change. Lorsque davantage de stations enregistrent de la pluie, le volume augmente.

In [ ]:
intensite = df["intensite_max"].to_numpy()
etendue = df["proportion_active"].to_numpy()

Afficher les courbes de pluie

In [ ]:
import matplotlib.pyplot as plt

# Convertir les dates en format datetime
df["date"] = pd.to_datetime(
    df["Date/heure locale (UTC+2)"]
)

# Créer deux graphiques superposés
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Graphique 1 : intensité maximale
axes[0].plot(df["date"], df["intensite_max"], color="blue")
axes[0].set_title("Intensité maximale de pluie à Bruxelles")
axes[0].set_ylabel("Pluie (mm/5 min)")
axes[0].grid(True)

# Graphique 2 : proportion de stations actives
axes[1].plot(df["date"], df["proportion_active"], color="green")
axes[1].set_title("Étendue des précipitations")
axes[1].set_ylabel("Proportion de stations actives")
axes[1].set_xlabel("Heure du 24 juillet 2025")

Préparer les données pour STRAUSS

In [ ]:
import numpy as np

# Nos deux indicateurs
intensite = df["intensite_max"].to_numpy(dtype=float)
etendue = df["proportion_active"].to_numpy(dtype=float)

# Temps normalisé entre 0 et 1
temps = np.linspace(0, 1, len(df))

# Normalisation de l'intensité avec une référence fixe
reference_pluie = 10.0  # mm/5 min
hauteur = np.clip(intensite / reference_pluie, 0, 1)

# Volume selon la proportion de stations actives
volume = np.clip(etendue, 0, 1)

print("Nombre d'instants :", len(temps))
print("Hauteur maximale normalisée :", hauteur.max())
print("Volume maximal :", volume.max())

Créer le son avec STRAUSS

In [ ]:
from strauss.sonification import Sonification
from strauss.sources import Objects
from strauss.score import Score
from strauss.generator import Synthesizer

# Durée de notre premier essai : 60 secondes
duree = 60

# Une seule note de référence
score = Score([["A2"]], duree)

# Choisir le synthétiseur
generator = Synthesizer()
generator.load_preset("pitch_mapper")

# Associer les données aux paramètres sonores
data_son = {
    "pitch": 1.0,
    "time_evo": temps,
    "pitch_shift": hauteur,
    "volume": volume,
    "azimuth": 0.5,
    "polar": 0.5
}

# Préparer la source sonore
sources = Objects(data_son.keys())
sources.fromdict(data_son)
sources.apply_mapping_functions()

# Générer la sonification
soni = Sonification(score, sources, generator, "stereo")
soni.render()

# Écouter dans le notebook
soni.notebook_display()

Ici en heaut on as réussi à représenter deux caractéristiques de la pluie : son intensité maximale et son étendue (encore à tester si le mapping est ok ou à modifier comme aussi l'intervalle de telps réalisé). Intensité maximale → hauteur. Proportion de stations actives → volume. On conserve les 24 heures